<a href="https://colab.research.google.com/github/Sayak-coder/spam_mssg_and_fake_news_detector_using_LSTM_and_Bidirectional_LSTM/blob/main/Fake_news_classification_using_Bidirectional_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('/content/news_dataset.csv')
df.head()

,label,text
0,REAL,Payal has accused filmmaker Anurag Kashyap of ...
1,FAKE,A four-minute-long video of a woman criticisin...
2,FAKE,"Republic Poll, a fake Twitter account imitatin..."
3,REAL,"Delhi teen finds place on UN green list, turns..."
4,REAL,Delhi: A high-level meeting underway at reside...


In [28]:
df['label']=df['label'].map({'FAKE':0,'REAL':1})

In [3]:
df.isna().sum()

,0
label,0
text,8


In [5]:
df.dropna(inplace=True)

In [8]:
df.shape

(3721, 2)

In [29]:
X=df.drop('label',axis=1)
y=df['label']

In [10]:
import tensorflow as tf
tf.__version__

'2.20.0'

In [11]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.layers import LSTM
from tensorflow.keras.layers import Dense

In [12]:
import nltk
from nltk.corpus import stopwords
import re

In [14]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [17]:
X.reset_index(inplace=True)

In [18]:
from nltk.stem.porter import PorterStemmer
stemmer=PorterStemmer()
corpus=[]
for i in range(0,len(X)):
  review=re.sub('[^a-zA-Z]',' ',X['text'][i])
  review=review.lower()
  review=review.split()

  review=(stemmer.stem(word) for word in review if not word in stopwords.words('english'))
  review=' '.join(review)
  corpus.append(review)

In [22]:
corpus

['payal accus filmmak anurag kashyap behav inappropri video went viral maintain stanc speak etim said want speak long time today final thought must get head tweet incid sometim ago metoo movement happen mani peopl told delet tweet els would stop get work manag advis remov tweet compli post anurag block whatsapp',
 'four minut long video woman criticis govern anti citizenship amend act ralli delhi earlier januari go viral fals claim woman show late prime minist atal bihari vajpaye niec caption hindi translat respect vajpaye ji niec final broken silenc listen say translat hindi also read muslim politician disguis hindu anti caa protest video come time protest citizenship amend act propos nation regist citizen gone unab month video woman seen say follow hindi british lot bad least outsid first land came afar came still differ indian govern british british educ illiter least british govern peopl understand indian talk india tell congress tell other tell us silent pakistan crazi talk pakist

In [23]:
voc_size=5000
one_hot_rep=[one_hot(words,voc_size) for words in corpus]

In [24]:
sentence_len=350
embedded_doc=pad_sequences(one_hot_rep,padding='pre',maxlen=sentence_len)

In [26]:
embedded_doc[3]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0, 4997,  209, 4092,  632,
       3131, 3149, 4653,  680, 3612, 2776, 3895, 4997, 2889, 2596, 2228,
        275,  680, 3612, 3123, 2776, 1614, 2606,  345, 4172,  391, 4606,
       4348, 3661, 3814, 3160,  634, 2981, 4127, 2338, 4284, 2240,  314,
       4348,  345, 3054, 1913, 1247, 4444, 4087, 31

In [27]:
embedding_vector_features=40
model=Sequential()
model.add(Embedding(voc_size,embedding_vector_features,input_length=sentence_len))
model.add(Bidirectional(LSTM(150)))
model.add(Dense(1,activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam',metrics=['accuracy'])
print(model.summary())

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [30]:
y.head()

,label
0,1
1,0
2,0
3,1
4,1


In [31]:
import numpy as np
X_final=np.array(embedded_doc)
y_final=np.array(y)

In [32]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X_final,y_final,test_size=0.2,random_state=45)

In [33]:
epochs=5
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=epochs,batch_size=32)

Epoch 1/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 89s 920ms/step - accuracy: 0.9110 - loss: 0.2684 - val_accuracy: 0.9973 - val_loss: 0.0219
Epoch 2/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 146s 965ms/step - accuracy: 0.9906 - loss: 0.0446 - val_accuracy: 0.9987 - val_loss: 0.0103
Epoch 3/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 90s 967ms/step - accuracy: 0.9940 - loss: 0.0311 - val_accuracy: 0.9973 - val_loss: 0.0161
Epoch 4/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 87s 933ms/step - accuracy: 0.9973 - loss: 0.0147 - val_accuracy: 0.9933 - val_loss: 0.0327
Epoch 5/5
93/93 ━━━━━━━━━━━━━━━━━━━━ 147s 977ms/step - accuracy: 0.9943 - loss: 0.0301 - val_accuracy: 0.9987 - val_loss: 0.0096


In [34]:
y_pred=model.predict(X_test)

24/24 ━━━━━━━━━━━━━━━━━━━━ 8s 310ms/step


In [35]:
y_pred=np.where(y_pred>0.6,1,0)

In [36]:
from sklearn.metrics import confusion_matrix,accuracy_score
print(confusion_matrix(y_test,y_pred))
print(accuracy_score(y_test,y_pred))

[[365   1]
 [  0 379]]
0.9986577181208054


In [37]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       366
           1       1.00      1.00      1.00       379

    accuracy                           1.00       745
   macro avg       1.00      1.00      1.00       745
weighted avg       1.00      1.00      1.00       745



```markdown
# Fake News Detection using LSTM

## Project Description
This project aims to build a deep learning model to detect fake news. The model is built using an LSTM (Long Short-Term Memory) network, a type of recurrent neural network particularly effective for sequential data like text. The process involves data loading, preprocessing, model creation, training, and evaluation.

## Dataset
The dataset used for this project is `news_dataset.csv`, which contains news articles and their corresponding labels (REAL or FAKE).

## Setup and Installation
To run this notebook, you'll need the following Python libraries. You can install them using pip:

```bash
pip install pandas numpy nltk tensorflow scikit-learn
```

## Data Preprocessing
1.  **Loading Data**: The `news_dataset.csv` file is loaded into a pandas DataFrame.
2.  **Handling Missing Values**: Missing values in the 'text' column are dropped.
3.  **Label Encoding**: The 'label' column (FAKE/REAL) is mapped to numerical values (0/1).
4.  **Text Cleaning and Stemming**:
    *   Special characters are removed, and text is converted to lowercase.
    *   Stopwords are removed using NLTK's English stopwords.
    *   Porter Stemmer is applied to reduce words to their root form.
5.  **One-Hot Representation**: Each word in the corpus is converted into a one-hot representation based on a defined vocabulary size.
6.  **Padding Sequences**: The one-hot encoded sequences are padded to a fixed length (`sentence_len`) to ensure uniform input for the LSTM model.

## Model Architecture
The model is a Sequential Keras model with the following layers:

*   **Embedding Layer**: Converts positive integers (word indices) into dense vectors of fixed size (`embedding_vector_features`).
*   **Bidirectional LSTM Layer**: A recurrent layer with 150 units, capable of learning long-term dependencies in the text sequences.
*   **Dense Output Layer**: A single neuron with a sigmoid activation function for binary classification (fake or real news).

**Model Summary:**
```
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
=================================================================
 embedding (Embedding)       (None, 350, 40)           200000    
                                                                 
 bidirectional (Bidirectional (None, 300)             229200    
)                                                                
                                                                 
 dense (Dense)               (None, 1)                 301       
                                                                 
=================================================================
Total params: 429501 (1.64 MB)
Trainable params: 429501 (1.64 MB)
Non-trainable params: 0 (0.00 B)
_________________________________________________________________
```

## Training
*   The dataset is split into training and testing sets (80% train, 20% test).
*   The model is compiled with `binary_crossentropy` loss, `adam` optimizer, and `accuracy` as the metric.
*   The model is trained for 5 epochs with a batch size of 32.

## Results
After training, the model's performance is evaluated using the confusion matrix, accuracy score, and classification report.

**Confusion Matrix:**
```
[[365   1]
 [  0 379]]
```

**Accuracy Score:**
```
0.9986577181208054
```

**Classification Report:**
```
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       366
           1       1.00      1.00      1.00       379

    accuracy                           1.00       745
   macro avg       1.00      1.00      1.00       745
weighted avg       1.00      1.00      1.00       745
```
The model achieved a high accuracy of approximately 99.87%, indicating strong performance in distinguishing between fake and real news articles on the given dataset.